### 例題
###以下のデータフロー図のテーブルを作成してください。

<img src="./sample_dataflow">

▪️使用テーブル
- samples.bakehouse.sales_transactions
- samples.bakehouse.sales_customers
- samples.bakehouse.sales_franchises

▪️補足
- 必要に応じて、カタログ・スキーマは自身で作成して下さい。
- ノートブックで作成して下さい。（言語は問いません）
- DeltaTableの形式で作成して下さい。


## 回答

## スキーマの作成

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.work;

## sales_analysisの作成

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.work.sales_analysis AS
SELECT sales_transactions.dateTime, 
  sales_transactions.product,
  sales_transactions.quantity, 
  sales_transactions.totalPrice,
  concat(sales_customers.first_name, '_', sales_customers.last_name) AS customer_name, 
  sales_customers.state, 
  sales_customers.country as customers_country, -- 問題のミスで項目countoryが二つだったので、別名にしています
  sales_customers.gender,
  sales_franchises.name as franchises_name,
  sales_franchises.city,
  sales_franchises.country as franchises_country -- 問題のミスで項目countoryが二つだったので、別名にしています
FROM samples.bakehouse.sales_transactions
LEFT JOIN samples.bakehouse.sales_customers ON sales_transactions.customerID = sales_customers.customerID
LEFT JOIN samples.bakehouse.sales_franchises ON sales_transactions.franchiseID = sales_franchises.franchiseID
;

In [0]:
%sql
SELECT * FROM workspace.work.sales_analysis

## sales_costomers_analysisの作成

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.work.sales_costomers_analysis AS
SELECT sales_analysis.customer_name,
  count(sales_analysis.customer_name) AS transaction_count,
  sum(sales_analysis.totalPrice) AS totalPrice_inLife
FROM workspace.work.sales_analysis
GROUP BY sales_analysis.customer_name
;

In [0]:
%sql
SELECT * FROM workspace.work.sales_costomers_analysis

## sales_product_analysisの作成

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.work.sales_product_analysis AS
SELECT sales_analysis.product,
  sum(sales_analysis.quantity) AS total_quantity,
  sum(sales_analysis.totalPrice) AS totalPrice_product
FROM workspace.work.sales_analysis
GROUP BY sales_analysis.product
;

In [0]:
%sql
SELECT * FROM workspace.work.sales_product_analysis